# Sigma-Hole Molecular Docking Pipeline - Google Colab Version

This notebook demonstrates how to use the Sigma-Hole docking pipeline for modeling halogen-bonding (σ-hole) interactions in Google Colab.

## Overview

The Sigma-Hole pipeline models directional halogen-bonding interactions using:
1. Dummy atoms (Extra Points) - Virtual charge sites positioned along the C–X bond axis
2. Vmax-based charges - Dummy charge calibrated from DFT-computed electrostatic potential maxima
3. Physics-based scoring - Lennard-Jones + Coulomb energy evaluation

## Setup

First, we need to install the required packages and upload our data files.

In [ ]:
# Install required packages
!pip install numpy pandas rdkit matplotlib seaborn py3Dmol pillow pydantic

# Check if we need to mount Google Drive (optional)
# from google.colab import drive
# drive.mount('/content/drive')

## File Upload

Upload your input files:
- CSV file with compound data (compound_id, smiles, halogen, vmax)
- Receptor file in PDBQT format
- Optional: ligand structure files (SDF, PDB, etc.)

For this demo, we'll use the example files included in the repository.

In [ ]:
# List uploaded files
import os
print("Files in current directory:")
for f in os.listdir('.'):
    if f.endswith(('.csv', '.pdbqt', '.sdf', '.pdb')):
        print(f"  {f}")

## Example: Running the Sigma-Hole Pipeline

Let's run a simple example using the included test data.

In [ ]:
# Import the pipeline
from sigma_hole_pipeline import SigmaHolePipeline

# Initialize the pipeline
pipeline = SigmaHolePipeline()

# Run the pipeline with example data
print("Starting Sigma-Hole pipeline...")
results = pipeline.run_full_pipeline(
    input_csv='test_input.csv',
    receptor_input='receptor.pdbqt',
    structure_dir='.',  # Current directory for structure files
    structure_ext='.sdf'
)

print("Pipeline completed!")
print(f"Results keys: {list(results.keys())}")

## Accessing Results

The pipeline returns a dictionary with various results. Let's examine the key outputs.

In [ ]:
# Check if docking results exist
if 'docking_results' in results:
    df = results['docking_results']
    print("Docking Results:")
    print(df.head())
    print(f"\nTotal compounds docked: {len(df)}")

# Check analysis results
if 'analysis' in results:
    analysis = results['analysis']
    print("\nAnalysis Results:")
    for key, value in analysis.items():
        if isinstance(value, pd.DataFrame):
            print(f"{key}: {value.shape[0]} rows")
            print(value.head())
        else:
            print(f"{key}: {value}")

## Visualizing Results

Let's create some basic visualizations of the docking results.

In [ ]:
# Plot binding energy distribution
if 'docking_results' in results and len(results['docking_results']) > 0:
    import matplotlib.pyplot as plt
    import seaborn as sns

    df = results['docking_results']
    plt.figure(figsize=(10, 6))
    sns.histplot(data=df, x='binding_energy_kcalmol', bins=20)
    plt.title('Binding Energy Distribution')
    plt.xlabel('Binding Energy (kcal/mol)')
    plt.ylabel('Frequency')
    plt.show()

# Show top hits
if 'analysis' in results and 'top_hits' in results['analysis']:
    top_hits = results['analysis']['top_hits']
    print("Top 5 Hits:")
    print(top_hits[['compound_id', 'binding_energy_kcalmol', 'halogen']].head())

## Advanced Usage: Using Pre-generated Structures

For best results, use pre-optimized DFT structures instead of generating from SMILES. This ensures Vmax values are consistent with the input geometry.

Example with DFT structures:
```python
results = pipeline.run_full_pipeline(
    input_csv='compounds_with_vmax.csv',
    receptor_input='receptor.pdbqt',
    structure_dir='./dft_structures',
    structure_ext='.sdf'
)
```

## Saving Results

You can save the results to files for further analysis or to download from Colab.

In [ ]:
# Save docking results to CSV
if 'docking_results' in results:
    results['docking_results'].to_csv('docking_results.csv', index=False)
    print("Docking results saved to 'docking_results.csv'")

# Save analysis results
if 'analysis' in results:
    for key, value in results['analysis'].items():
        if isinstance(value, pd.DataFrame):
            value.to_csv(f'analysis_{key}.csv', index=False)
            print(f"Analysis {key} saved to 'analysis_{key}.csv'")

## Next Steps

1. Upload your own CSV file with compound data
2. Upload your receptor in PDBQT format
3. (Optional) Upload pre-optimized ligand structures for best accuracy
4. Run the pipeline and analyze the results
5. Download the results files from Colab

## References

For more information about the Sigma-Hole method and halogen bonding, see:
- Politzer, P., et al. (2013). Halogen bonding: An interaction divided. *CrystEngComm*, 15(16), 3029-3039.
- Cavallo, G., et al. (2016). The halogen bond. *Chemical Reviews*, 116(4), 2478-2601.
- Kolář, M. H., et al. (2019). σ-Hole interaction parameters. *Journal of chemical theory and computation*, 15(5), 2972-2984.